# Workers' Compensation Claims Valuation & Risk Triage
**Course Project:** Data Mining & Machine Learning (DMML)  
**Authors:** Marius Occhiminuti · Gaëtan Amiens  
**Dataset:** Workers' Compensation Claims (100,000 Records)


## 1. Imports & Global Settings

In [ ]:
import os
import warnings
from time import perf_counter
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import RobustScaler, OneHotEncoder
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.dummy import DummyRegressor
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from lightgbm import LGBMRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", None)

RANDOM_STATE = 42
TEST_SIZE = 0.20
print("Libraries imported successfully!")


## 2. Load Dataset & Feature Engineering
Dataset is loaded once into memory to speed up iteration.


In [ ]:
def resolve_data_path(csv_filename: str = "workers_comp.csv") -> str:
    script_dir = Path.cwd()
    candidates = [
        Path(csv_filename),
        script_dir / csv_filename,
        script_dir.parent / csv_filename,
        script_dir / "src" / csv_filename
    ]
    for p in candidates:
        if p.exists():
            return str(p)
    checked = "\n".join([f" - {p.resolve()}" for p in candidates])
    raise FileNotFoundError(f"Dataset '{csv_filename}' not found. Checked:\n{checked}")

data_path = resolve_data_path("workers_comp.csv")
print(f"Loading dataset from: {data_path}")
df = pd.read_csv(data_path)

# Datetime Parsing & Feature Engineering (W1)
df['DateTimeOfAccident'] = pd.to_datetime(df['DateTimeOfAccident'])
df['DateReported'] = pd.to_datetime(df['DateReported'])

# Report delay in days
df['ReportDelayDays'] = (df['DateReported'] - df['DateTimeOfAccident']).dt.total_seconds() / (24 * 3600)
df['ReportDelayDays'] = df['ReportDelayDays'].clip(lower=0)

# Temporal features
df['AccidentYear'] = df['DateTimeOfAccident'].dt.year
df['AccidentMonth'] = df['DateTimeOfAccident'].dt.month

# Handle missing text values
if 'ClaimDescription' in df.columns:
    df['ClaimDescription'] = df['ClaimDescription'].fillna("")

print(f"Dataset loaded: {df.shape[0]} rows, {df.shape[1]} columns.")
df.head(3)


## 3. Exploratory Data Analysis (EDA)
Inspecting target distribution `UltimateIncurredClaimCost` and heavy-tail log-transformation.


In [ ]:
plt.figure(figsize=(12, 4))

plt.subplot(1, 2, 1)
sns.histplot(df['UltimateIncurredClaimCost'], bins=50, kde=True, color='crimson')
plt.title('Target Distribution: UltimateIncurredClaimCost ($)')
plt.xlabel('Cost ($)')

plt.subplot(1, 2, 2)
sns.histplot(np.log1p(df['UltimateIncurredClaimCost']), bins=50, kde=True, color='navy')
plt.title('Log-Transformed Target: log(1 + Cost)')
plt.xlabel('log(1 + Cost)')

plt.tight_layout()
plt.show()


## 4. Leak-Proof Preprocessor & Train/Test Split (W3/W4 & W10)
Setting up a leak-proof `ColumnTransformer` wrapping numerical, categorical, and text TF-IDF features.


In [ ]:
target_col = 'UltimateIncurredClaimCost'
numeric_features = [
    'Age', 'WeeklyPay', 'HoursWorkedPerWeek', 'DaysWorkedPerWeek', 
    'DependentChildren', 'DependentsOther', 'ReportDelayDays', 'InitialCaseEstimate'
]
categorical_features = ['Gender', 'MaritalStatus', 'PartTimeFullTime']
text_feature = 'ClaimDescription' if 'ClaimDescription' in df.columns else None

feature_cols = numeric_features + categorical_features + ([text_feature] if text_feature else [])
X = df[feature_cols]
y = df[target_col]

# Train/Test Split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=TEST_SIZE, random_state=RANDOM_STATE
)

print(f"Train Set: {X_train.shape[0]} rows | Test Set: {X_test.shape[0]} rows")

# ColumnTransformer
numeric_transformer = Pipeline(steps=[('scaler', RobustScaler())])
categorical_transformer = Pipeline(steps=[('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False))])

transformers = [
    ('num', numeric_transformer, numeric_features),
    ('cat', categorical_transformer, categorical_features)
]

if text_feature:
    text_transformer = Pipeline(steps=[
        ('tfidf', TfidfVectorizer(max_features=300, stop_words='english', ngram_range=(1, 2)))
    ])
    transformers.append(('text', text_transformer, text_feature))

preprocessor = ColumnTransformer(transformers=transformers)
print("Preprocessor initialized!")


## 5. Model Zoo Benchmark (W1-W10)
Evaluating Linear Baselines, Random Forest, Gradient Boosting, and LightGBM with target log-transformation and prediction clipping.


In [ ]:
def evaluate_regression_models(models: dict, X_train: pd.DataFrame, X_test: pd.DataFrame, 
                               y_train: pd.Series, y_test: pd.Series) -> pd.DataFrame:
    results = []
    max_log_val = np.log1p(y_train).max()
    min_log_val = np.log1p(y_train).min()
    
    for name, model in models.items():
        print(f"Fitting {name}...")
        start_time = perf_counter()
        
        # Fit on log-transformed target
        model.fit(X_train, np.log1p(y_train))
        fit_time = perf_counter() - start_time
        
        # Predict & clip to log-bounds
        raw_log_preds = model.predict(X_test)
        clipped_log_preds = np.clip(raw_log_preds, a_min=min_log_val, a_max=max_log_val)
        
        # Expm1 back to dollars
        y_pred = np.expm1(clipped_log_preds)
        
        rmse = np.sqrt(mean_squared_error(y_test, y_pred))
        mae = mean_absolute_error(y_test, y_pred)
        r2 = r2_score(y_test, y_pred)
        valid = y_test > 0
        mape = np.mean(np.abs((y_test[valid] - y_pred[valid]) / y_test[valid])) * 100
        
        results.append({
            "model": name,
            "rmse": round(rmse, 2),
            "mae": round(mae, 2),
            "mape_pct": round(mape, 2),
            "r2_score": round(r2, 4),
            "fit_time_sec": round(fit_time, 4)
        })
        
    return pd.DataFrame(results).sort_values("rmse").reset_index(drop=True)

models = {
    "dummy_mean_baseline": Pipeline([('prep', preprocessor), ('reg', DummyRegressor(strategy='mean'))]),
    "linear_regression": Pipeline([('prep', preprocessor), ('reg', LinearRegression())]),
    "ridge_regression": Pipeline([('prep', preprocessor), ('reg', Ridge(alpha=10.0))]),
    "lasso_regression": Pipeline([('prep', preprocessor), ('reg', Lasso(alpha=1.0))]),
    "random_forest": Pipeline([('prep', preprocessor), ('reg', RandomForestRegressor(n_estimators=100, max_depth=12, random_state=RANDOM_STATE, n_jobs=-1))]),
    "gradient_boosting": Pipeline([('prep', preprocessor), ('reg', GradientBoostingRegressor(n_estimators=100, learning_rate=0.1, max_depth=5, random_state=RANDOM_STATE))]),
    "lightgbm": Pipeline([('prep', preprocessor), ('reg', LGBMRegressor(n_estimators=100, learning_rate=0.1, max_depth=6, random_state=RANDOM_STATE, verbose=-1))])
}

benchmark_df = evaluate_regression_models(models, X_train, X_test, y_train, y_test)
benchmark_df


## 6. Benchmark Visualization
Plotting model performance comparisons across MAE and RMSE.


In [ ]:
plt.figure(figsize=(10, 5))
sns.barplot(data=benchmark_df, x='model', y='mae', palette='Blues_r')
plt.title('MAE ($) Comparison Across Models')
plt.ylabel('Mean Absolute Error ($)')
plt.xticks(rotation=30, ha='right')
plt.grid(True, linestyle='--', alpha=0.6)
plt.tight_layout()
plt.show()
